In [1]:
# ============================================================
# CELL 1 — INSTALLS
# ============================================================

%pip install -q torch torchvision qai-hub onnx onnxruntime

import json
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.models.video as video_models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")
print(f"   PyTorch: {torch.__version__}")

Note: you may need to restart the kernel to use updated packages.
✅ Device: cpu
   PyTorch: 2.11.0+cpu


In [2]:
# ============================================================
# CELL 3 — CONFIG  (update CKPT_PATH from Cell 2 output)
# ============================================================

CKPT_PATH    = "C:\\Users\\USER\\Downloads\\qualcom results\\89%\\best_r2plus1d_qevd.pth"
MAPPING_PATH = "C:\\Users\\USER\\Downloads\\qualcom results\\89%\\class_mapping.json"
ONNX_PATH    = "C:\\Users\\USER\\Downloads\\qualcom results\\89%\\qualcomm_r2plus1d.onnx"
OUTPUT_DIR   = "C:\\Users\\USER\\Downloads\\qualcom results\\results\\89%"

# Competition constants — never change
NUM_FRAMES   = 16
FRAME_SIZE   = 112
LATENCY_LIMIT_MS = 34.0
AIHUB_DEVICE =  "Dragonwing IQ-9075 EVK"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load class mapping from previous notebook
with open(MAPPING_PATH) as f:
    mapping = json.load(f)

# Support both formats:
# 1) {"classes": [...]}
# 2) {"root": {"classes": [...], ...}}
if "classes" in mapping:
    CLASSES = mapping["classes"]
elif "root" in mapping and "classes" in mapping["root"]:
    CLASSES = mapping["root"]["classes"]
else:
    raise KeyError("Could not find 'classes' in mapping JSON. Expected 'classes' or 'root.classes'.")

NUM_CLASSES = len(CLASSES)

print(f"✅ Paths configured")
print(f"   Checkpoint : {CKPT_PATH}")
print(f"   Classes    : {NUM_CLASSES}")

✅ Paths configured
   Checkpoint : C:\Users\USER\Downloads\qualcom results\89%\best_r2plus1d_qevd.pth
   Classes    : 91


In [3]:
# ============================================================
# CELL 4 — REBUILD MODEL & LOAD WEIGHTS
# ============================================================

def build_model(num_classes, backbone_name):
    if backbone_name == "r2plus1d_18":
        model = video_models.r2plus1d_18(weights=None)
    elif backbone_name == "mc3_18":
        model = video_models.mc3_18(weights=None)
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    in_feat  = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_feat, num_classes),
    )
    return model

# Load checkpoint
# Resolve checkpoint path (handles relative path issues)
ckpt_candidates = [
    CKPT_PATH,
    os.path.join(OUTPUT_DIR, os.path.basename(CKPT_PATH)),
    os.path.join(os.path.dirname(MAPPING_PATH), os.path.basename(CKPT_PATH)),
]
ckpt_file = next((p for p in ckpt_candidates if os.path.isfile(p)), None)

if ckpt_file is None:
    raise FileNotFoundError(
        f"Checkpoint not found. Tried: {ckpt_candidates}\n"
        f"Current working dir: {os.getcwd()}"
    )

# Load checkpoint (compatible with different torch versions / checkpoint formats)
try:
    raw_ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=True)
except TypeError:
    raw_ckpt = torch.load(ckpt_file, map_location="cpu")

# Extract the actual state_dict from common checkpoint formats
if isinstance(raw_ckpt, dict) and "model_state" in raw_ckpt and isinstance(raw_ckpt["model_state"], dict):
    state_dict = raw_ckpt["model_state"]
elif isinstance(raw_ckpt, dict) and "state_dict" in raw_ckpt and isinstance(raw_ckpt["state_dict"], dict):
    state_dict = raw_ckpt["state_dict"]
elif isinstance(raw_ckpt, dict) and "model" in raw_ckpt and isinstance(raw_ckpt["model"], dict):
    state_dict = raw_ckpt["model"]
elif isinstance(raw_ckpt, dict):
    state_dict = raw_ckpt
else:
    raise ValueError(f"Unsupported checkpoint format in: {ckpt_file}")

# Remove DataParallel prefix if present
state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in state_dict.items()}

ckpt = {
    "model_state": state_dict,
    "epoch": raw_ckpt.get("epoch", -1) if isinstance(raw_ckpt, dict) else -1,
    "val_acc": raw_ckpt.get("val_acc", float("nan")) if isinstance(raw_ckpt, dict) else float("nan"),
}

# Pick backbone from checkpoint file name (your file indicates r2plus1d)
ckpt_name = os.path.basename(ckpt_file).lower()
backbone_name = "r2plus1d_18" if "r2plus1d" in ckpt_name else "mc3_18"

print(f"✅ Using checkpoint: {ckpt_file}")
print(f"✅ Backbone selected: {backbone_name}")

model = build_model(NUM_CLASSES, backbone_name)
# Handle checkpoint heads saved as a plain Linear layer
if "fc.weight" in ckpt["model_state"] and "fc.1.weight" not in ckpt["model_state"]:
    ckpt["model_state"]["fc.1.weight"] = ckpt["model_state"].pop("fc.weight")
    ckpt["model_state"]["fc.1.bias"]   = ckpt["model_state"].pop("fc.bias")

model.load_state_dict(ckpt["model_state"], strict=True)
model.eval()

print(f"✅ Model loaded")
print(f"   Trained epochs : {ckpt['epoch']}")
print(f"   Best val acc   : {ckpt['val_acc']:.2f}%")
print(f"   Num classes    : {NUM_CLASSES}")

# Sanity check output shape
with torch.no_grad():
    dummy = torch.randn(1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)
    out   = model(dummy)
    print(f"   Output shape   : {out.shape}  ✓")
    del dummy, out

✅ Using checkpoint: C:\Users\USER\Downloads\qualcom results\89%\best_r2plus1d_qevd.pth
✅ Backbone selected: r2plus1d_18
✅ Model loaded
   Trained epochs : 13
   Best val acc   : 89.14%
   Num classes    : 91
   Output shape   : torch.Size([1, 91])  ✓


In [4]:
!pip install -q onnx onnxruntime onnxscript qai-hub


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\USER\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
# ============================================================
# CELL 5 — EXPORT TO ONNX  (replace previous Cell 5)
# ============================================================

import onnx
from onnx.external_data_helper import convert_model_to_external_data, load_external_data_for_model
import onnxruntime as ort

print("📦 Exporting to ONNX ...")

dummy = torch.randn(1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)

# Export to a temp path first
ONNX_TMP = "C:\\Users\\USER\\Downloads\\qualcom results\\89%\\qualcomm_r2plus1d.onnx"

# Use legacy exporter to avoid requiring onnxscript
torch.onnx.export(
    model,
    dummy,
    ONNX_TMP,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes=None,
    verbose=False,
    dynamo=False,   # <- fixes ModuleNotFoundError: onnxscript
)
print("✅ Initial export done")

# ── Inline all external weights into single file ──────────────
print("🔧 Inlining external weights ...")
onnx_model = onnx.load(ONNX_TMP, load_external_data=True)

# Save as single self-contained file — no external data files
onnx.save(
    onnx_model,
    ONNX_PATH,
    save_as_external_data = False,   # ← forces everything inline
)

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f"✅ ONNX saved → {ONNX_PATH}  ({size_mb:.2f} MB)")

# ── Verify ────────────────────────────────────────────────────
onnx.checker.check_model(ONNX_PATH)
print("✅ ONNX checker passed")

# ── Parity check ──────────────────────────────────────────────
with torch.no_grad():
    torch_out = model(dummy).numpy()

sess    = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
ort_out = sess.run(None, {"input": dummy.numpy()})[0]
max_diff = float(np.abs(torch_out - ort_out).max())
print(f"✅ Max PyTorch ↔ ONNX diff : {max_diff:.6f}  "
      f"{'✓ OK' if max_diff < 1e-3 else '⚠ check this'}")


📦 Exporting to ONNX ...


C:\Users\USER\AppData\Local\Temp\ipykernel_19736\2281472406.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ Initial export done
🔧 Inlining external weights ...
✅ ONNX saved → C:\Users\USER\Downloads\qualcom results\89%\qualcomm_r2plus1d.onnx  (119.55 MB)
✅ ONNX checker passed
✅ Max PyTorch ↔ ONNX diff : 0.000005  ✓ OK


In [6]:
# Quick version diagnostic — run this alone first
import qai_hub as hub
print(f"qai_hub version : {hub.__version__}")
print(f"Available attrs : {[x for x in dir(hub) if not x.startswith('_')]}")

qai_hub version : 0.47.0
Available attrs : ['Client', 'ClientConfig', 'CompileJob', 'CompileJobResult', 'CompileJobSummary', 'Dataset', 'Device', 'Error', 'Framework', 'InferenceJob', 'InferenceJobResult', 'InferenceJobSummary', 'InputSpecs', 'InternalError', 'Job', 'JobArtifactType', 'JobResult', 'JobStatus', 'JobSummary', 'JobType', 'LinkJob', 'LinkJobResult', 'LinkJobSummary', 'Model', 'ModelMetadataKey', 'ProfileJob', 'ProfileJobResult', 'ProfileJobSummary', 'Project', 'QuantizeDtype', 'QuantizeJob', 'QuantizeJobResult', 'QuantizeJobSummary', 'SourceModelType', 'UserError', 'api_status_codes', 'api_utils', 'client', 'get_dataset', 'get_datasets', 'get_device_attributes', 'get_devices', 'get_frameworks', 'get_job', 'get_job_summaries', 'get_model', 'get_models', 'hub', 'public_api_pb2', 'public_rest_api', 'set_session_token', 'set_verbose', 'submit_compile_and_link_jobs', 'submit_compile_and_profile_jobs', 'submit_compile_job', 'submit_inference_job', 'submit_link_job', 'submit_prof

In [7]:
# ── Quick diagnostic — run this alone ────────────────────────
all_devices = hub.get_devices()
# Filter for Dragonwing
matches = [d for d in all_devices if "dragon" in d.name.lower() or "iq" in d.name.lower()]
print("Dragonwing devices found:")
for d in matches:
    print(f"  '{d.name}'")

# Also print all device names if no matches
if not matches:
    print("\nNo Dragonwing found. All devices:")
    for d in all_devices:
        print(f"  '{d.name}'")

Dragonwing devices found:
  'Snapdragon X Elite CRD'
  'Snapdragon X Plus 8-Core CRD'
  'Snapdragon X2 Elite CRD'
  'Snapdragon 8 Elite QRD'
  'Snapdragon 8 Elite Gen 5 QRD'
  'Snapdragon 7 Gen 4 QRD'
  'Dragonwing Q-6690 MTP'
  'Dragonwing RB3 Gen 2 Vision Kit'
  'Dragonwing IQ-9075 EVK'


In [8]:
# ============================================================
# CELL 6 — COMPILE ON QUALCOMM AI HUB  (replace previous Cell 6)
# ============================================================

import importlib, qai_hub
importlib.reload(qai_hub)          # clear any stale config from previous runs

try:
    from kaggle_secrets import UserSecretsClient
except ModuleNotFoundError:
    UserSecretsClient = None

def _get_qai_hub_token():
    # 1) Standard env vars
    for key in ("QAI_HUB_TOKEN", "QAIHUB_TOKEN"):
        val = os.getenv(key)
        if val and val.strip():
            return val.strip()

    # 2) Kaggle secrets (if available)
    if UserSecretsClient is not None:
        client = UserSecretsClient()
        for key in ("QAI_HUB_TOKEN", "QAIHUB_TOKEN", "esh3gtugsxb9xn9jdr32bfraeb0ws7na2bzsqkus"):
            try:
                val = client.get_secret(key)
                if val and str(val).strip():
                    return str(val).strip()
            except Exception:
                pass

    # 3) Interactive fallback (works in local Jupyter)
    import getpass
    val = getpass.getpass("Enter Qualcomm AI Hub token: ").strip()
    if val:
        return val

    raise RuntimeError(
        "Missing Qualcomm AI Hub token. Set QAI_HUB_TOKEN/QAIHUB_TOKEN or paste it when prompted."
    )

token = _get_qai_hub_token()

# ── Auth — correct for qai_hub 0.47.0 ────────────────────────
os.environ["QAI_HUB_TOKEN"] = token
hub = qai_hub.Client()
print("✅ Authenticated with Qualcomm AI Hub")

# ── Verify ────────────────────────────────────────────────────
devices = hub.get_devices()
print(f"   {len(devices)} devices available")

# ── Device ───────────────────────────────────────────────────
# This will now work correctly
device = hub.get_devices(name=AIHUB_DEVICE)[0]
print(f"🎯 Target device : {device.name}")

# ── Upload ───────────────────────────────────────────────────
print("⬆️  Uploading ONNX model ...")
hub_model = hub.upload_model(ONNX_PATH)
print(f"✅ Uploaded : {hub_model}")

# ── Compile ──────────────────────────────────────────────────
print("⚙️  Compiling ...")
compile_job = hub.submit_compile_job(
    model       = hub_model,
    device      = device,
    name        = "LPCVC2026_Track2_MC3_18",
    options     = "--target_runtime qnn_context_binary", 
    input_specs = {"input": (1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)},
)
print(f"✅ Compile job : {compile_job.job_id}")
print("⏳ Waiting ...")
compile_job.wait()

status = compile_job.get_status()
if not status.success:
    raise RuntimeError(f"❌ Compile failed: {status.message}")

print("🎉 Compilation succeeded!")
target_model = compile_job.get_target_model()

✅ Authenticated with Qualcomm AI Hub
   81 devices available
🎯 Target device : Dragonwing IQ-9075 EVK
⬆️  Uploading ONNX model ...
Uploading qualcomm_r2plus1d.onnx


100%|██████████| 120M/120M [01:19<00:00, 1.58MB/s] 


✅ Uploaded : Model(model_id='mqv2x1p8n', name='qualcomm_r2plus1d.onnx')
⚙️  Compiling ...
Scheduled compile job (j563dj075) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j563dj075/

✅ Compile job : j563dj075
⏳ Waiting ...
Waiting for compile job (j563dj075) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
🎉 Compilation succeeded!


In [9]:
# ============================================================
# CELL 7 — PROFILE LATENCY  (replace previous Cell 7)
# ============================================================

import json, glob

print("⏱️  Profiling on device ...")
profile_job = hub.submit_profile_job(
    model  = target_model,
    device = device,
    name   = "LPCVC2026_Track2_MC3_18_profile",
)
print(f"✅ Profile job : {profile_job.job_id}")
print("⏳ Waiting ...")
profile_job.wait()

profile_dir = "C:\\Users\\USER\\Downloads\\qualcom results\\88%"
os.makedirs(profile_dir, exist_ok=True)
profile_job.download_results(artifacts_dir=profile_dir)

# ── Read from saved JSON ──────────────────────────────────────
json_files = glob.glob(f"{profile_dir}/*.json")
with open(json_files[-1]) as f:        # latest file
    profile_data = json.load(f)

# estimated_inference_time is in MICROSECONDS → convert to ms
inf_time_us = profile_data["execution_summary"]["estimated_inference_time"]
inf_time    = inf_time_us / 1000.0     # microseconds → milliseconds

# Also grab useful stats
all_times_ms = [t / 1000 for t in
                profile_data["execution_summary"]["all_inference_times"]]
peak_mem_mb  = profile_data["execution_summary"][
                "estimated_inference_peak_memory"] / 1024 / 1024

print("\n" + "=" * 55)
print(f"⚡ Inference time (best)  : {inf_time:.2f} ms")
print(f"⚡ Inference time (mean)  : {sum(all_times_ms)/len(all_times_ms):.2f} ms")
print(f"⚡ Inference time (max)   : {max(all_times_ms):.2f} ms")
print(f"💾 Peak memory            : {peak_mem_mb:.2f} MB")
print(f"🎯 Limit                  : {LATENCY_LIMIT_MS} ms")
print(f"📊 Margin (best)          : {LATENCY_LIMIT_MS - inf_time:.2f} ms")
if inf_time < LATENCY_LIMIT_MS:
    print(f"✅ VALID — all {len(all_times_ms)} runs under 34ms!")
else:
    print(f"❌ OVER LIMIT by {inf_time - LATENCY_LIMIT_MS:.2f} ms")
print("=" * 55)

⏱️  Profiling on device ...
Scheduled profile job (jp34w3rzg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp34w3rzg/

✅ Profile job : jp34w3rzg
⏳ Waiting ...
Waiting for profile job (jp34w3rzg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


jp34w3rzg_runtime.log: 100%|██████████| 35.7k/35.7k [00:00<00:00, 1.84MB/s]


Saved profile results to C:\Users\USER\Downloads\qualcom results\88%\LPCVC2026_Track2_MC3_18_profile_jp34w3rzg_results.json

⚡ Inference time (best)  : 27.96 ms
⚡ Inference time (mean)  : 29.28 ms
⚡ Inference time (max)   : 29.72 ms
💾 Peak memory            : 8.61 MB
🎯 Limit                  : 34.0 ms
📊 Margin (best)          : 6.04 ms
✅ VALID — all 100 runs under 34ms!
